# 03 — Bike Sharing Exploratory Data Analysis
This notebook explores temporal, calendar, seasonal, weather-related, and user-segment patterns in bike-sharing demand.
The analysis uses the prepared daily and hourly datasets created during the data-cleaning stage. The main objective is to identify operationally meaningful differences between casual and registered users.

## 1. Load and Validate Clean Data

This section loads the prepared daily and hourly datasets and performs a minimal validation before exploratory analysis.

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
notebook_dir = Path.cwd()
project_root = notebook_dir.parents[1]

clean_data_dir = project_root / "01_data" / "clean"
tables_output_dir = project_root / "04_outputs" / "tables"

figures_output_dir = (
    project_root
    / "04_outputs"
    / "figures"
    / "exploratory"
)

day_clean_file = clean_data_dir / "day_clean.csv"
hour_clean_file = clean_data_dir / "hour_clean.csv"

tables_output_dir.mkdir(
    parents=True,
    exist_ok=True
)

figures_output_dir.mkdir(
    parents=True,
    exist_ok=True
)

In [ ]:
day_data = pd.read_csv(
    day_clean_file,
    parse_dates=['dteday']
)

hour_data = pd.read_csv(
    hour_clean_file,
    parse_dates=[
        'dteday',
        'datetime'
    ]
)

In [ ]:
data_loading_summary = pd.DataFrame({
    'dataset': [
        'day',
        'hour'
    ],

    'row_count': [
        len(day_data),
        len(hour_data)
    ],

    'column_count': [
        day_data.shape[1],
        hour_data.shape[1]
    ],

    'minimum_date': [
        day_data['dteday'].min(),
        hour_data['dteday'].min()
    ],

    'maximum_date': [
        day_data['dteday'].max(),
        hour_data['dteday'].max()
    ],

    'date_is_datetime': [
        pd.api.types.is_datetime64_any_dtype(
            day_data['dteday']
        ),
        pd.api.types.is_datetime64_any_dtype(
            hour_data['dteday']
        )
    ]
})

display(data_loading_summary)

In [ ]:
hour_datetime_validation = pd.DataFrame({
    'datetime_is_valid': [
        pd.api.types.is_datetime64_any_dtype(
            hour_data['datetime']
        )
    ],

    'minimum_datetime': [
        hour_data['datetime'].min()
    ],

    'maximum_datetime': [
        hour_data['datetime'].max()
    ],

    'missing_datetime': [
        hour_data['datetime'].isna().sum()
    ]
})

display(hour_datetime_validation)

### Clean data loading result

The prepared daily and hourly datasets were loaded successfully.

The expected number of observations and date coverage were preserved, and the date and hourly timestamp variables were restored as datetime data types.

The datasets are ready for exploratory analysis.

## 2. Overall Demand Overview

This section summarizes the overall rental volume, daily demand level, and the contribution of casual and registered users.

In [ ]:
overall_demand_summary = pd.DataFrame({
    'metric': [
        'observation days',
        'total rentals',
        'average daily rentals',
        'median daily rentals',
        'minimum daily rentals',
        'maximum daily rentals'
    ],

    'value': [
        day_data['dteday'].nunique(),
        day_data['cnt'].sum(),
        day_data['cnt'].mean(),
        day_data['cnt'].median(),
        day_data['cnt'].min(),
        day_data['cnt'].max()
    ]
})

overall_demand_summary['value'] = (
    overall_demand_summary['value']
    .round(2)
)

display(overall_demand_summary)

In [ ]:
total_rentals = day_data['cnt'].sum()

user_segment_summary = pd.DataFrame({
    'user_segment': [
        'casual',
        'registered'
    ],

    'total_rentals': [
        day_data['casual'].sum(),
        day_data['registered'].sum()
    ]
})

user_segment_summary['share_pct'] = (
    user_segment_summary['total_rentals']
    / total_rentals
    * 100
).round(2)

display(user_segment_summary)

In [ ]:
minimum_demand_day = (
    day_data.loc[
        day_data['cnt'].idxmin(),
        [
            'dteday',
            'cnt',
            'casual',
            'registered',
            'season_label',
            'weather_label'
        ]
    ]
    .to_frame(name='minimum_demand_day')
)

maximum_demand_day = (
    day_data.loc[
        day_data['cnt'].idxmax(),
        [
            'dteday',
            'cnt',
            'casual',
            'registered',
            'season_label',
            'weather_label'
        ]
    ]
    .to_frame(name='maximum_demand_day')
)

demand_extreme_days = pd.concat(
    [
        minimum_demand_day,
        maximum_demand_day
    ],
    axis=1
)

display(demand_extreme_days)

In [ ]:
overall_demand_summary.to_csv(
    tables_output_dir / 'overall_demand_summary.csv',
    index=False
)

user_segment_summary.to_csv(
    tables_output_dir / 'user_segment_summary.csv',
    index=False
)

### Overall demand overview result

The dataset provides a complete two-year view of daily bike-sharing demand.

Registered users account for the majority of total rentals, while casual users represent a smaller but operationally distinct portion of demand.

The difference between the average and median daily demand, together with the observed minimum and maximum values, indicates that demand levels vary substantially across the study period. These variations will be examined through year, month, season, calendar, hourly, and weather-related analyses.

## 3. Annual Demand Growth

This section compares bike-sharing demand between 2011 and 2012.

Both total rentals and average daily rentals are examined because 2012 contains one more calendar day than 2011. Casual and registered users are analyzed separately to identify changes in the composition of demand.

In [ ]:
yearly_demand_summary = (
    day_data
    .groupby('year')
    .agg(
        days_observed=('dteday', 'nunique'),
        total_rentals=('cnt', 'sum'),
        average_daily_rentals=('cnt', 'mean'),
        median_daily_rentals=('cnt', 'median'),
        casual_total=('casual', 'sum'),
        casual_daily_average=('casual', 'mean'),
        registered_total=('registered', 'sum'),
        registered_daily_average=('registered', 'mean')
    )
    .reset_index()
)

In [ ]:
yearly_demand_summary['casual_share_pct'] = (
    yearly_demand_summary['casual_total']
    / yearly_demand_summary['total_rentals']
    * 100
)

yearly_demand_summary['registered_share_pct'] = (
    yearly_demand_summary['registered_total']
    / yearly_demand_summary['total_rentals']
    * 100
)

In [ ]:
yearly_demand_display = yearly_demand_summary.round({
    'average_daily_rentals': 2,
    'median_daily_rentals': 2,
    'casual_daily_average': 2,
    'registered_daily_average': 2,
    'casual_share_pct': 2,
    'registered_share_pct': 2
})

display(yearly_demand_display)

In [ ]:
yearly_by_year = (
    yearly_demand_summary
    .set_index('year')
)

In [ ]:
growth_metrics = [
    'total_rentals',
    'average_daily_rentals',
    'casual_total',
    'casual_daily_average',
    'registered_total',
    'registered_daily_average'
]

In [ ]:
annual_growth_summary = pd.DataFrame({
    'metric': growth_metrics,

    'value_2011': (
        yearly_by_year
        .loc[2011, growth_metrics]
        .to_numpy()
    ),

    'value_2012': (
        yearly_by_year
        .loc[2012, growth_metrics]
        .to_numpy()
    )
})

In [ ]:
annual_growth_summary['absolute_change'] = (
    annual_growth_summary['value_2012']
    - annual_growth_summary['value_2011']
)

annual_growth_summary['growth_pct'] = (
    annual_growth_summary['absolute_change']
    / annual_growth_summary['value_2011']
    * 100
)

In [ ]:
annual_growth_display = annual_growth_summary.round({
    'value_2011': 2,
    'value_2012': 2,
    'absolute_change': 2,
    'growth_pct': 2
})

display(annual_growth_display)

In [ ]:
user_share_by_year = yearly_demand_summary[
    [
        'year',
        'casual_share_pct',
        'registered_share_pct'
    ]
].copy()

user_share_by_year = user_share_by_year.round(2)

display(user_share_by_year)

In [ ]:
annual_user_average = (
    yearly_demand_summary
    .set_index('year')
    [
        [
            'casual_daily_average',
            'registered_daily_average'
        ]
    ]
)

In [ ]:
ax = annual_user_average.plot(
    kind='bar',
    figsize=(8, 5)
)

ax.set_title(
    'Average Daily Rentals by User Segment and Year'
)

ax.set_xlabel('Year')
ax.set_ylabel('Average Daily Rentals')

ax.set_xticklabels(
    ax.get_xticklabels(),
    rotation=0
)

ax.legend(
    [
        'Casual',
        'Registered'
    ],
    title='User Segment'
)

plt.tight_layout()

plt.savefig(
    figures_output_dir
    / 'average_daily_rentals_by_year_and_user.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
yearly_demand_summary.to_csv(
    tables_output_dir / 'yearly_demand_summary.csv',
    index=False
)

annual_growth_summary.to_csv(
    tables_output_dir / 'annual_growth_summary.csv',
    index=False
)

user_share_by_year.to_csv(
    tables_output_dir / 'user_share_by_year.csv',
    index=False
)

### Annual demand growth result

Bike-sharing demand changed substantially between 2011 and 2012.

The comparison of average daily rentals confirms that the observed annual difference is not explained solely by the additional calendar day in 2012. Both casual and registered users should be evaluated separately because their growth rates and contributions to total demand are not necessarily identical.

The annual comparison describes a change in recorded demand, but it does not by itself establish the causes of that change. Seasonal, monthly, calendar, hourly, and weather-related patterns are examined in the following sections.

## 4. Monthly and Seasonal Demand Patterns

This section examines how bike-sharing demand changes across months and seasons.

Average daily rentals are used as the main comparison measure because months and seasons contain different numbers of observed days. The two years are also examined separately to avoid confusing seasonal patterns with annual demand growth.

In [ ]:
month_map = {
    1: 'january',
    2: 'february',
    3: 'march',
    4: 'april',
    5: 'may',
    6: 'june',
    7: 'july',
    8: 'august',
    9: 'september',
    10: 'october',
    11: 'november',
    12: 'december'
}

In [ ]:
monthly_demand_summary = (
    day_data
    .groupby(
        [
            'year',
            'mnth'
        ]
    )
    .agg(
        days_observed=('dteday', 'nunique'),
        total_rentals=('cnt', 'sum'),
        average_daily_rentals=('cnt', 'mean'),
        median_daily_rentals=('cnt', 'median'),
        casual_daily_average=('casual', 'mean'),
        registered_daily_average=('registered', 'mean')
    )
    .reset_index()
)

monthly_demand_summary['month_name'] = (
    monthly_demand_summary['mnth']
    .map(month_map)
)

monthly_demand_summary = monthly_demand_summary[
    [
        'year',
        'mnth',
        'month_name',
        'days_observed',
        'total_rentals',
        'average_daily_rentals',
        'median_daily_rentals',
        'casual_daily_average',
        'registered_daily_average'
    ]
]

monthly_demand_display = monthly_demand_summary.round({
    'average_daily_rentals': 2,
    'median_daily_rentals': 2,
    'casual_daily_average': 2,
    'registered_daily_average': 2
})

display(monthly_demand_display)

In [ ]:
monthly_average_pivot = (
    monthly_demand_summary
    .pivot(
        index='mnth',
        columns='year',
        values='average_daily_rentals'
    )
    .sort_index()
)

display(
    monthly_average_pivot.round(2)
)

In [ ]:
ax = monthly_average_pivot.plot(
    kind='line',
    marker='o',
    figsize=(10, 5)
)

ax.set_title(
    'Average Daily Bike Rentals by Month and Year'
)

ax.set_xlabel('Month')
ax.set_ylabel('Average Daily Rentals')

ax.set_xticks(
    range(1, 13)
)

ax.set_xticklabels(
    [
        'Jan',
        'Feb',
        'Mar',
        'Apr',
        'May',
        'Jun',
        'Jul',
        'Aug',
        'Sep',
        'Oct',
        'Nov',
        'Dec'
    ]
)

ax.legend(
    title='Year'
)

plt.tight_layout()

plt.savefig(
    figures_output_dir
    / 'average_daily_rentals_by_month_and_year.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
season_order = [
    'spring',
    'summer',
    'fall',
    'winter'
]

seasonal_demand_summary = (
    day_data
    .groupby(
        [
            'year',
            'season_label'
        ]
    )
    .agg(
        days_observed=('dteday', 'nunique'),
        total_rentals=('cnt', 'sum'),
        average_daily_rentals=('cnt', 'mean'),
        median_daily_rentals=('cnt', 'median'),
        casual_daily_average=('casual', 'mean'),
        registered_daily_average=('registered', 'mean')
    )
    .reset_index()
)

season_order_map = {
    season: position
    for position, season in enumerate(
        season_order,
        start=1
    )
}

seasonal_demand_summary['season_order'] = (
    seasonal_demand_summary['season_label']
    .map(season_order_map)
)

seasonal_demand_summary = (
    seasonal_demand_summary
    .sort_values(
        [
            'year',
            'season_order'
        ]
    )
    .drop(
        columns='season_order'
    )
    .reset_index(drop=True)
)

seasonal_demand_display = seasonal_demand_summary.round({
    'average_daily_rentals': 2,
    'median_daily_rentals': 2,
    'casual_daily_average': 2,
    'registered_daily_average': 2
})

display(seasonal_demand_display)

In [ ]:
seasonal_average_pivot = (
    seasonal_demand_summary
    .pivot(
        index='season_label',
        columns='year',
        values='average_daily_rentals'
    )
    .reindex(season_order)
)

display(
    seasonal_average_pivot.round(2)
)

In [ ]:
ax = seasonal_average_pivot.plot(
    kind='bar',
    figsize=(9, 5)
)

ax.set_title(
    'Average Daily Bike Rentals by Season and Year'
)

ax.set_xlabel('Season')
ax.set_ylabel('Average Daily Rentals')

ax.set_xticklabels(
    [
        'Spring',
        'Summer',
        'Fall',
        'Winter'
    ],
    rotation=0
)

ax.legend(
    title='Year'
)

plt.tight_layout()

plt.savefig(
    figures_output_dir
    / 'average_daily_rentals_by_season_and_year.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
seasonal_user_summary = (
    day_data
    .groupby('season_label')
    .agg(
        casual_daily_average=('casual', 'mean'),
        registered_daily_average=('registered', 'mean')
    )
    .reindex(season_order)
    .reset_index()
)

seasonal_user_summary = seasonal_user_summary.round(2)

display(seasonal_user_summary)

In [ ]:
seasonal_user_plot = (
    seasonal_user_summary
    .set_index('season_label')
)

ax = seasonal_user_plot.plot(
    kind='bar',
    figsize=(9, 5)
)

ax.set_title(
    'Average Daily Rentals by Season and User Segment'
)

ax.set_xlabel('Season')
ax.set_ylabel('Average Daily Rentals')

ax.set_xticklabels(
    [
        'Spring',
        'Summer',
        'Fall',
        'Winter'
    ],
    rotation=0
)

ax.legend(
    [
        'Casual',
        'Registered'
    ],
    title='User Segment'
)

plt.tight_layout()

plt.savefig(
    figures_output_dir
    / 'average_daily_rentals_by_season_and_user.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
monthly_demand_summary.to_csv(
    tables_output_dir
    / 'monthly_demand_summary.csv',
    index=False
)

seasonal_demand_summary.to_csv(
    tables_output_dir
    / 'seasonal_demand_summary.csv',
    index=False
)

seasonal_user_summary.to_csv(
    tables_output_dir
    / 'seasonal_user_summary.csv',
    index=False
)

### Monthly and seasonal demand analysis result

Monthly and seasonal demand patterns were examined using average daily rentals to control for differences in the number of calendar days.

The years were analyzed separately so that seasonal variation would not be confused with the overall growth in demand between 2011 and 2012. Casual and registered users were also summarized separately because their seasonal patterns may differ.

These results describe recurring temporal associations. They do not establish that the season or month alone caused the observed changes in demand.

## 5. Weekday, Working-Day, and Holiday Patterns

This section examines demand across weekdays, working days, non-working days, and holidays.

Casual and registered users are analyzed separately because calendar conditions may affect the two user segments differently.

In [ ]:
weekday_order = [
    'sunday',
    'monday',
    'tuesday',
    'wednesday',
    'thursday',
    'friday',
    'saturday'
]

In [ ]:
weekday_demand_summary = (
    day_data
    .groupby('weekday_label')
    .agg(
        days_observed=('dteday', 'nunique'),
        total_rentals=('cnt', 'sum'),
        average_daily_rentals=('cnt', 'mean'),
        median_daily_rentals=('cnt', 'median'),
        casual_daily_average=('casual', 'mean'),
        registered_daily_average=('registered', 'mean')
    )
    .reindex(weekday_order)
    .reset_index()
)

In [ ]:
weekday_demand_summary['casual_share_pct'] = (
    weekday_demand_summary['casual_daily_average']
    / weekday_demand_summary['average_daily_rentals']
    * 100
)

In [ ]:
weekday_demand_display = weekday_demand_summary.round({
    'average_daily_rentals': 2,
    'median_daily_rentals': 2,
    'casual_daily_average': 2,
    'registered_daily_average': 2,
    'casual_share_pct': 2
})

display(weekday_demand_display)

In [ ]:
weekday_user_plot = (
    weekday_demand_summary
    .set_index('weekday_label')
    [
        [
            'casual_daily_average',
            'registered_daily_average'
        ]
    ]
)

In [ ]:
ax = weekday_user_plot.plot(
    kind='line',
    marker='o',
    figsize=(10, 5)
)

ax.set_title(
    'Average Daily Rentals by Weekday and User Segment'
)

ax.set_xlabel('Weekday')
ax.set_ylabel('Average Daily Rentals')

ax.set_xticks(
    range(len(weekday_order))
)

ax.set_xticklabels(
    [
        'Sun',
        'Mon',
        'Tue',
        'Wed',
        'Thu',
        'Fri',
        'Sat'
    ]
)

ax.legend(
    [
        'Casual',
        'Registered'
    ],
    title='User Segment'
)

plt.tight_layout()

plt.savefig(
    figures_output_dir
    / 'average_daily_rentals_by_weekday_and_user.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
workingday_demand_summary = (
    day_data
    .groupby('workingday')
    .agg(
        days_observed=('dteday', 'nunique'),
        total_rentals=('cnt', 'sum'),
        average_daily_rentals=('cnt', 'mean'),
        median_daily_rentals=('cnt', 'median'),
        casual_daily_average=('casual', 'mean'),
        registered_daily_average=('registered', 'mean')
    )
    .reset_index()
)

In [ ]:
workingday_demand_summary['day_type'] = (
    workingday_demand_summary['workingday']
    .map({
        0: 'non_working_day',
        1: 'working_day'
    })
)

In [ ]:
workingday_demand_summary = workingday_demand_summary[
    [
        'workingday',
        'day_type',
        'days_observed',
        'total_rentals',
        'average_daily_rentals',
        'median_daily_rentals',
        'casual_daily_average',
        'registered_daily_average'
    ]
]

In [ ]:
workingday_demand_display = (
    workingday_demand_summary
    .round({
        'average_daily_rentals': 2,
        'median_daily_rentals': 2,
        'casual_daily_average': 2,
        'registered_daily_average': 2
    })
)

display(workingday_demand_display)

In [ ]:
workingday_user_plot = (
    workingday_demand_summary
    .set_index('day_type')
    [
        [
            'casual_daily_average',
            'registered_daily_average'
        ]
    ]
)

In [ ]:
ax = workingday_user_plot.plot(
    kind='bar',
    figsize=(8, 5)
)

ax.set_title(
    'Average Daily Rentals by Day Type and User Segment'
)

ax.set_xlabel('Day Type')
ax.set_ylabel('Average Daily Rentals')

ax.set_xticklabels(
    [
        'Non-Working Day',
        'Working Day'
    ],
    rotation=0
)

ax.legend(
    [
        'Casual',
        'Registered'
    ],
    title='User Segment'
)

plt.tight_layout()

plt.savefig(
    figures_output_dir
    / 'average_daily_rentals_by_workingday_and_user.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
holiday_demand_summary = (
    day_data
    .groupby('holiday')
    .agg(
        days_observed=('dteday', 'nunique'),
        total_rentals=('cnt', 'sum'),
        average_daily_rentals=('cnt', 'mean'),
        median_daily_rentals=('cnt', 'median'),
        casual_daily_average=('casual', 'mean'),
        registered_daily_average=('registered', 'mean')
    )
    .reset_index()
)

In [ ]:
holiday_demand_summary['holiday_type'] = (
    holiday_demand_summary['holiday']
    .map({
        0: 'non_holiday',
        1: 'holiday'
    })
)

In [ ]:
holiday_demand_summary = holiday_demand_summary[
    [
        'holiday',
        'holiday_type',
        'days_observed',
        'total_rentals',
        'average_daily_rentals',
        'median_daily_rentals',
        'casual_daily_average',
        'registered_daily_average'
    ]
]

holiday_demand_display = holiday_demand_summary.round({
    'average_daily_rentals': 2,
    'median_daily_rentals': 2,
    'casual_daily_average': 2,
    'registered_daily_average': 2
})

display(holiday_demand_display)

In [ ]:
weekday_demand_summary.to_csv(
    tables_output_dir
    / 'weekday_demand_summary.csv',
    index=False
)

workingday_demand_summary.to_csv(
    tables_output_dir
    / 'workingday_demand_summary.csv',
    index=False
)

holiday_demand_summary.to_csv(
    tables_output_dir
    / 'holiday_demand_summary.csv',
    index=False
)

### Calendar pattern analysis result

Bike-sharing demand was compared across weekdays, working days, non-working days, and official holidays.

Casual and registered users were examined separately because the overall demand level can conceal differences in user composition. The working-day comparison should not be interpreted as a simple weekday-versus-weekend comparison because non-working days include both weekends and official holidays.

Holiday results should also be interpreted cautiously because the number of observed holidays is substantially smaller than the number of non-holiday days. The hourly analysis will provide additional evidence for interpreting whether the observed user-segment patterns are consistent with commuting or leisure-related behavior.

## 6. Hourly Demand Patterns

This section examines bike-sharing demand across the 24 hours of the day.

Hourly patterns are analyzed for casual and registered users separately and are also compared between working and non-working days. Missing hourly records are not treated as zero demand.

In [ ]:
hourly_demand_summary = (
    hour_data
    .groupby('hr')
    .agg(
        observed_records=('datetime', 'count'),
        average_hourly_rentals=('cnt', 'mean'),
        median_hourly_rentals=('cnt', 'median'),
        casual_hourly_average=('casual', 'mean'),
        registered_hourly_average=('registered', 'mean')
    )
    .reindex(range(24))
    .reset_index()
)

hourly_demand_display = hourly_demand_summary.round({
    'average_hourly_rentals': 2,
    'median_hourly_rentals': 2,
    'casual_hourly_average': 2,
    'registered_hourly_average': 2
})

display(hourly_demand_display)

In [ ]:
hourly_user_plot = (
    hourly_demand_summary
    .set_index('hr')
    [
        [
            'casual_hourly_average',
            'registered_hourly_average'
        ]
    ]
)

In [ ]:
ax = hourly_user_plot.plot(
    kind='line',
    marker='o',
    figsize=(11, 5)
)

ax.set_title(
    'Average Hourly Rentals by User Segment'
)

ax.set_xlabel('Hour of Day')
ax.set_ylabel('Average Hourly Rentals')

ax.set_xticks(
    range(24)
)

ax.legend(
    [
        'Casual',
        'Registered'
    ],
    title='User Segment'
)

plt.tight_layout()

plt.savefig(
    figures_output_dir
    / 'average_hourly_rentals_by_user_segment.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
peak_hour_summary = pd.DataFrame({
    'demand_measure': [
        'total demand',
        'casual users',
        'registered users'
    ],

    'peak_hour': [
        hourly_demand_summary.loc[
            hourly_demand_summary[
                'average_hourly_rentals'
            ].idxmax(),
            'hr'
        ],

        hourly_demand_summary.loc[
            hourly_demand_summary[
                'casual_hourly_average'
            ].idxmax(),
            'hr'
        ],

        hourly_demand_summary.loc[
            hourly_demand_summary[
                'registered_hourly_average'
            ].idxmax(),
            'hr'
        ]
    ],

    'peak_average_rentals': [
        hourly_demand_summary[
            'average_hourly_rentals'
        ].max(),

        hourly_demand_summary[
            'casual_hourly_average'
        ].max(),

        hourly_demand_summary[
            'registered_hourly_average'
        ].max()
    ]
})

peak_hour_summary['peak_average_rentals'] = (
    peak_hour_summary['peak_average_rentals']
    .round(2)
)

display(peak_hour_summary)

In [ ]:
hourly_day_type_summary = (
    hour_data
    .groupby(
        [
            'workingday',
            'hr'
        ]
    )
    .agg(
        observed_records=('datetime', 'count'),
        average_hourly_rentals=('cnt', 'mean'),
        casual_hourly_average=('casual', 'mean'),
        registered_hourly_average=('registered', 'mean')
    )
    .reset_index()
)

hourly_day_type_summary['day_type'] = (
    hourly_day_type_summary['workingday']
    .map({
        0: 'non_working_day',
        1: 'working_day'
    })
)

hourly_day_type_display = hourly_day_type_summary.round({
    'average_hourly_rentals': 2,
    'casual_hourly_average': 2,
    'registered_hourly_average': 2
})

display(hourly_day_type_display)

In [ ]:
registered_hourly_by_day_type = (
    hourly_day_type_summary
    .pivot(
        index='hr',
        columns='day_type',
        values='registered_hourly_average'
    )
    .reindex(range(24))
)

In [ ]:
ax = registered_hourly_by_day_type.plot(
    kind='line',
    marker='o',
    figsize=(11, 5)
)

ax.set_title(
    'Registered User Demand by Hour and Day Type'
)

ax.set_xlabel('Hour of Day')
ax.set_ylabel('Average Hourly Registered Rentals')

ax.set_xticks(
    range(24)
)

ax.legend(
    [
        'Non-Working Day',
        'Working Day'
    ],
    title='Day Type'
)

plt.tight_layout()

plt.savefig(
    figures_output_dir
    / 'registered_hourly_demand_by_day_type.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
casual_hourly_by_day_type = (
    hourly_day_type_summary
    .pivot(
        index='hr',
        columns='day_type',
        values='casual_hourly_average'
    )
    .reindex(range(24))
)

In [ ]:
ax = casual_hourly_by_day_type.plot(
    kind='line',
    marker='o',
    figsize=(11, 5)
)

ax.set_title(
    'Casual User Demand by Hour and Day Type'
)

ax.set_xlabel('Hour of Day')
ax.set_ylabel('Average Hourly Casual Rentals')

ax.set_xticks(
    range(24)
)

ax.legend(
    [
        'Non-Working Day',
        'Working Day'
    ],
    title='Day Type'
)

plt.tight_layout()

plt.savefig(
    figures_output_dir
    / 'casual_hourly_demand_by_day_type.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
peak_hour_by_day_type = (
    hourly_day_type_summary
    .loc[
        hourly_day_type_summary
        .groupby('day_type')[
            'average_hourly_rentals'
        ]
        .idxmax(),
        [
            'day_type',
            'hr',
            'average_hourly_rentals'
        ]
    ]
    .rename(
        columns={
            'hr': 'peak_hour',
            'average_hourly_rentals':
                'peak_average_total_rentals'
        }
    )
    .reset_index(drop=True)
)

peak_hour_by_day_type[
    'peak_average_total_rentals'
] = (
    peak_hour_by_day_type[
        'peak_average_total_rentals'
    ]
    .round(2)
)

display(peak_hour_by_day_type)

In [ ]:
hourly_demand_summary.to_csv(
    tables_output_dir
    / 'hourly_demand_summary.csv',
    index=False
)

peak_hour_summary.to_csv(
    tables_output_dir
    / 'peak_hour_summary.csv',
    index=False
)

hourly_day_type_summary.to_csv(
    tables_output_dir
    / 'hourly_demand_by_day_type.csv',
    index=False
)

peak_hour_by_day_type.to_csv(
    tables_output_dir
    / 'peak_hour_by_day_type.csv',
    index=False
)

### Hourly demand analysis result

Bike-sharing demand varies substantially across the hours of the day, and the hourly profiles of casual and registered users are not identical.

The comparison between working and non-working days provides additional context for interpreting these differences. Patterns concentrated around morning and evening periods may be consistent with regular travel behavior, while broader midday patterns may be consistent with more flexible or leisure-oriented use.

These interpretations remain behavioral inferences because the dataset does not directly record trip purpose. The influence of incomplete hourly coverage will be assessed through a separate sensitivity analysis.

## 7. User Segment Composition

This section examines how the composition of bike-sharing demand changes across day types and hours of the day.

User-segment shares are calculated from aggregated rental totals rather than from the average of row-level percentages.

In [ ]:
segment_mix_by_day_type = (
    day_data
    .groupby('workingday')
    .agg(
        days_observed=('dteday', 'nunique'),
        casual_rentals=('casual', 'sum'),
        registered_rentals=('registered', 'sum'),
        total_rentals=('cnt', 'sum')
    )
    .reset_index()
)

segment_mix_by_day_type['day_type'] = (
    segment_mix_by_day_type['workingday']
    .map({
        0: 'non_working_day',
        1: 'working_day'
    })
)

In [ ]:
segment_mix_by_day_type['casual_share_pct'] = (
    segment_mix_by_day_type['casual_rentals']
    / segment_mix_by_day_type['total_rentals']
    * 100
)

segment_mix_by_day_type['registered_share_pct'] = (
    segment_mix_by_day_type['registered_rentals']
    / segment_mix_by_day_type['total_rentals']
    * 100
)

In [ ]:
segment_mix_by_day_type = segment_mix_by_day_type[
    [
        'workingday',
        'day_type',
        'days_observed',
        'casual_rentals',
        'registered_rentals',
        'total_rentals',
        'casual_share_pct',
        'registered_share_pct'
    ]
]

segment_mix_by_day_type_display = (
    segment_mix_by_day_type
    .round({
        'casual_share_pct': 2,
        'registered_share_pct': 2
    })
)

display(segment_mix_by_day_type_display)

In [ ]:
hourly_segment_mix = (
    hour_data
    .groupby('hr')
    .agg(
        observed_records=('datetime', 'count'),
        casual_rentals=('casual', 'sum'),
        registered_rentals=('registered', 'sum'),
        total_rentals=('cnt', 'sum')
    )
    .reindex(range(24))
    .reset_index()
)

In [ ]:
hourly_segment_mix['casual_share_pct'] = (
    hourly_segment_mix['casual_rentals']
    / hourly_segment_mix['total_rentals']
    * 100
)

hourly_segment_mix['registered_share_pct'] = (
    hourly_segment_mix['registered_rentals']
    / hourly_segment_mix['total_rentals']
    * 100
)

In [ ]:
hourly_segment_mix_display = hourly_segment_mix.round({
    'casual_share_pct': 2,
    'registered_share_pct': 2
})

display(hourly_segment_mix_display)

In [ ]:
highest_casual_share_row = hourly_segment_mix.loc[
    hourly_segment_mix['casual_share_pct'].idxmax()
]

highest_registered_share_row = hourly_segment_mix.loc[
    hourly_segment_mix['registered_share_pct'].idxmax()
]

segment_share_extremes = pd.DataFrame({
    'user_segment': [
        'casual',
        'registered'
    ],

    'highest_share_hour': [
        int(highest_casual_share_row['hr']),
        int(highest_registered_share_row['hr'])
    ],

    'highest_share_pct': [
        highest_casual_share_row['casual_share_pct'],
        highest_registered_share_row['registered_share_pct']
    ],

    'total_rentals_at_hour': [
        int(highest_casual_share_row['total_rentals']),
        int(highest_registered_share_row['total_rentals'])
    ]
})

segment_share_extremes['highest_share_pct'] = (
    segment_share_extremes['highest_share_pct']
    .round(2)
)

display(segment_share_extremes)

In [ ]:
hourly_segment_mix_by_day_type = (
    hour_data
    .groupby(
        [
            'workingday',
            'hr'
        ]
    )
    .agg(
        observed_records=('datetime', 'count'),
        casual_rentals=('casual', 'sum'),
        registered_rentals=('registered', 'sum'),
        total_rentals=('cnt', 'sum')
    )
    .reset_index()
)

In [ ]:
hourly_segment_mix_by_day_type['day_type'] = (
    hourly_segment_mix_by_day_type['workingday']
    .map({
        0: 'non_working_day',
        1: 'working_day'
    })
)

In [ ]:
hourly_segment_mix_by_day_type['casual_share_pct'] = (
    hourly_segment_mix_by_day_type['casual_rentals']
    / hourly_segment_mix_by_day_type['total_rentals']
    * 100
)

In [ ]:
hourly_segment_mix_by_day_type_display = (
    hourly_segment_mix_by_day_type
    .round({
        'casual_share_pct': 2
    })
)

display(hourly_segment_mix_by_day_type_display)

In [ ]:
casual_share_by_hour_and_day_type = (
    hourly_segment_mix_by_day_type
    .pivot(
        index='hr',
        columns='day_type',
        values='casual_share_pct'
    )
    .reindex(range(24))
)

In [ ]:
ax = casual_share_by_hour_and_day_type.plot(
    kind='line',
    marker='o',
    figsize=(11, 5)
)

ax.set_title(
    'Casual User Share by Hour and Day Type'
)

ax.set_xlabel('Hour of Day')
ax.set_ylabel('Casual Share of Rentals (%)')

ax.set_xticks(
    range(24)
)

ax.legend(
    [
        'Non-Working Day',
        'Working Day'
    ],
    title='Day Type'
)

plt.tight_layout()

plt.savefig(
    figures_output_dir
    / 'casual_user_share_by_hour_and_day_type.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
segment_mix_by_day_type.to_csv(
    tables_output_dir
    / 'segment_mix_by_day_type.csv',
    index=False
)

hourly_segment_mix.to_csv(
    tables_output_dir
    / 'hourly_segment_mix.csv',
    index=False
)

segment_share_extremes.to_csv(
    tables_output_dir
    / 'segment_share_extremes.csv',
    index=False
)

hourly_segment_mix_by_day_type.to_csv(
    tables_output_dir
    / 'hourly_segment_mix_by_day_type.csv',
    index=False
)

### User segment composition result

The composition of bike-sharing demand varies across day types and hours of the day.

User-segment shares provide information that is not visible from rental counts alone. A high segment share does not necessarily indicate high operational demand, because the total number of rentals at that hour may still be small.

Differences between working and non-working days provide additional evidence that casual and registered users follow distinct usage patterns. However, these patterns should not be interpreted as direct evidence of trip purpose because the dataset does not contain individual trip-purpose information.

## 8. Weather and Demand Relationships

This section examines the associations between bike-sharing demand and weather conditions.

The daily dataset is used so that each calendar day receives equal weight. Weather categories and continuous weather measurements are analyzed separately for total, casual, and registered demand.

The results are descriptive and should not be interpreted as evidence that weather conditions alone caused the observed demand differences.

In [ ]:
weather_order = [
    'clear_or_partly_cloudy',
    'mist_or_cloudy',
    'light_rain_or_snow',
    'heavy_rain_snow_or_fog'
]

weather_order_map = {
    weather: position
    for position, weather in enumerate(
        weather_order,
        start=1
    )
}

In [ ]:
weather_condition_summary = (
    day_data
    .groupby('weather_label')
    .agg(
        days_observed=('dteday', 'nunique'),
        total_rentals=('cnt', 'sum'),
        average_daily_rentals=('cnt', 'mean'),
        median_daily_rentals=('cnt', 'median'),
        casual_daily_average=('casual', 'mean'),
        registered_daily_average=('registered', 'mean')
    )
    .reset_index()
)

weather_condition_summary['weather_order'] = (
    weather_condition_summary['weather_label']
    .map(weather_order_map)
)

weather_condition_summary = (
    weather_condition_summary
    .sort_values('weather_order')
    .drop(columns='weather_order')
    .reset_index(drop=True)
)

In [ ]:
weather_condition_display = (
    weather_condition_summary
    .round({
        'average_daily_rentals': 2,
        'median_daily_rentals': 2,
        'casual_daily_average': 2,
        'registered_daily_average': 2
    })
)

display(weather_condition_display)

In [ ]:
weather_by_year_summary = (
    day_data
    .groupby(
        [
            'year',
            'weather_label'
        ]
    )
    .agg(
        days_observed=('dteday', 'nunique'),
        average_daily_rentals=('cnt', 'mean'),
        casual_daily_average=('casual', 'mean'),
        registered_daily_average=('registered', 'mean')
    )
    .reset_index()
)

weather_by_year_summary['weather_order'] = (
    weather_by_year_summary['weather_label']
    .map(weather_order_map)
)

weather_by_year_summary = (
    weather_by_year_summary
    .sort_values(
        [
            'year',
            'weather_order'
        ]
    )
    .drop(columns='weather_order')
    .reset_index(drop=True)
)

weather_by_year_display = weather_by_year_summary.round({
    'average_daily_rentals': 2,
    'casual_daily_average': 2,
    'registered_daily_average': 2
})

display(weather_by_year_display)

In [ ]:
weather_user_plot = (
    weather_condition_summary
    .set_index('weather_label')
    [
        [
            'casual_daily_average',
            'registered_daily_average'
        ]
    ]
)

In [ ]:
ax = weather_user_plot.plot(
    kind='bar',
    figsize=(10, 5)
)

ax.set_title(
    'Average Daily Rentals by Weather Condition and User Segment'
)

ax.set_xlabel('Weather Condition')
ax.set_ylabel('Average Daily Rentals')

ax.set_xticklabels(
    [
        label.replace('_', ' ').title()
        for label in weather_user_plot.index
    ],
    rotation=20,
    ha='right'
)

ax.legend(
    [
        'Casual',
        'Registered'
    ],
    title='User Segment'
)

plt.tight_layout()

plt.savefig(
    figures_output_dir
    / 'average_daily_rentals_by_weather_and_user.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
weather_variables = [
    'temp_c',
    'atemp_c',
    'humidity_pct',
    'windspeed_original_scale'
]

demand_variables = [
    'cnt',
    'casual',
    'registered'
]

correlation_rows = []

for weather_variable in weather_variables:
    for demand_variable in demand_variables:
        correlation_rows.append({
            'weather_variable': weather_variable,
            'demand_variable': demand_variable,
            'observations': len(day_data),

            'pearson_correlation': (
                day_data[weather_variable]
                .corr(
                    day_data[demand_variable],
                    method='pearson'
                )
            ),

            'spearman_correlation': (
                day_data[weather_variable]
                .corr(
                    day_data[demand_variable],
                    method='spearman'
                )
            )
        })

weather_correlation_summary = pd.DataFrame(
    correlation_rows
)

weather_correlation_summary[
    [
        'pearson_correlation',
        'spearman_correlation'
    ]
] = (
    weather_correlation_summary[
        [
            'pearson_correlation',
            'spearman_correlation'
        ]
    ]
    .round(3)
)

display(weather_correlation_summary)

In [ ]:
temperature_redundancy_check = pd.DataFrame({
    'variable_pair': [
        'temp_c and atemp_c'
    ],

    'pearson_correlation': [
        day_data['temp_c'].corr(
            day_data['atemp_c'],
            method='pearson'
        )
    ],

    'spearman_correlation': [
        day_data['temp_c'].corr(
            day_data['atemp_c'],
            method='spearman'
        )
    ]
})

temperature_redundancy_check[
    [
        'pearson_correlation',
        'spearman_correlation'
    ]
] = (
    temperature_redundancy_check[
        [
            'pearson_correlation',
            'spearman_correlation'
        ]
    ]
    .round(3)
)

display(temperature_redundancy_check)

In [ ]:
ax = day_data.plot.scatter(
    x='temp_c',
    y='cnt',
    alpha=0.5,
    figsize=(9, 5)
)

ax.set_title(
    'Daily Bike Rentals versus Temperature'
)

ax.set_xlabel('Temperature (°C)')
ax.set_ylabel('Daily Rentals')

plt.tight_layout()

plt.savefig(
    figures_output_dir
    / 'daily_rentals_vs_temperature.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
ax = day_data.plot.scatter(
    x='humidity_pct',
    y='cnt',
    alpha=0.5,
    figsize=(9, 5)
)

ax.set_title(
    'Daily Bike Rentals versus Humidity'
)

ax.set_xlabel('Humidity (%)')
ax.set_ylabel('Daily Rentals')

plt.tight_layout()

plt.savefig(
    figures_output_dir
    / 'daily_rentals_vs_humidity.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
ax = day_data.plot.scatter(
    x='windspeed_original_scale',
    y='cnt',
    alpha=0.5,
    figsize=(9, 5)
)

ax.set_title(
    'Daily Bike Rentals versus Wind Speed'
)

ax.set_xlabel('Wind Speed — Original Scale')
ax.set_ylabel('Daily Rentals')

plt.tight_layout()

plt.savefig(
    figures_output_dir
    / 'daily_rentals_vs_windspeed.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
weather_condition_summary.to_csv(
    tables_output_dir
    / 'weather_condition_summary.csv',
    index=False
)

weather_by_year_summary.to_csv(
    tables_output_dir
    / 'weather_by_year_summary.csv',
    index=False
)

weather_correlation_summary.to_csv(
    tables_output_dir
    / 'weather_correlation_summary.csv',
    index=False
)

temperature_redundancy_check.to_csv(
    tables_output_dir
    / 'temperature_redundancy_check.csv',
    index=False
)

### Weather relationship analysis result

Bike-sharing demand was compared across categorical and continuous weather conditions.

The analysis included the number of observed days in each weather category because weather groups are not equally represented. Pearson and Spearman correlations were used to distinguish linear relationships from more general monotonic relationships.

Temperature, apparent temperature, humidity, and wind speed were examined as associated variables rather than isolated causal drivers. Calendar conditions, season, annual demand growth, and other unobserved factors may also contribute to the observed patterns.

The zero-humidity observation and incomplete hourly coverage will be evaluated through sensitivity checks before final conclusions are reported.

## 9. Data Quality Sensitivity Checks

This section evaluates whether the identified zero-humidity observation and incomplete hourly dates materially affect the main exploratory results.

The original datasets remain unchanged. Alternative subsets are created only for comparison.

In [ ]:
day_without_zero_humidity = (
    day_data.loc[
        day_data['zero_humidity_flag'] == 0
    ]
    .copy()
)

hour_complete_days = (
    hour_data.loc[
        hour_data['incomplete_day_flag'] == 0
    ]
    .copy()
)

In [ ]:
sensitivity_sample_summary = pd.DataFrame({
    'analysis_sample': [
        'all daily records',
        'daily records without zero humidity',
        'all hourly records',
        'hourly records from complete days'
    ],

    'record_count': [
        len(day_data),
        len(day_without_zero_humidity),
        len(hour_data),
        len(hour_complete_days)
    ],

    'date_count': [
        day_data['dteday'].nunique(),
        day_without_zero_humidity['dteday'].nunique(),
        hour_data['dteday'].nunique(),
        hour_complete_days['dteday'].nunique()
    ]
})

display(sensitivity_sample_summary)

In [ ]:
humidity_sensitivity_rows = []

for demand_variable in [
    'cnt',
    'casual',
    'registered'
]:
    full_pearson = (
        day_data['humidity_pct']
        .corr(
            day_data[demand_variable],
            method='pearson'
        )
    )

    filtered_pearson = (
        day_without_zero_humidity['humidity_pct']
        .corr(
            day_without_zero_humidity[demand_variable],
            method='pearson'
        )
    )

    full_spearman = (
        day_data['humidity_pct']
        .corr(
            day_data[demand_variable],
            method='spearman'
        )
    )

    filtered_spearman = (
        day_without_zero_humidity['humidity_pct']
        .corr(
            day_without_zero_humidity[demand_variable],
            method='spearman'
        )
    )

    humidity_sensitivity_rows.append({
        'demand_variable': demand_variable,
        'pearson_all_records': full_pearson,
        'pearson_without_zero': filtered_pearson,
        'pearson_change': (
            filtered_pearson - full_pearson
        ),
        'spearman_all_records': full_spearman,
        'spearman_without_zero': filtered_spearman,
        'spearman_change': (
            filtered_spearman - full_spearman
        )
    })

humidity_sensitivity_summary = pd.DataFrame(
    humidity_sensitivity_rows
)

humidity_sensitivity_summary[
    [
        'pearson_all_records',
        'pearson_without_zero',
        'pearson_change',
        'spearman_all_records',
        'spearman_without_zero',
        'spearman_change'
    ]
] = (
    humidity_sensitivity_summary[
        [
            'pearson_all_records',
            'pearson_without_zero',
            'pearson_change',
            'spearman_all_records',
            'spearman_without_zero',
            'spearman_change'
        ]
    ]
    .round(4)
)

display(humidity_sensitivity_summary)

In [ ]:
hourly_all_summary = (
    hour_data
    .groupby('hr')
    .agg(
        observed_records_all=('datetime', 'count'),
        total_average_all=('cnt', 'mean'),
        casual_average_all=('casual', 'mean'),
        registered_average_all=('registered', 'mean')
    )
    .reindex(range(24))
    .reset_index()
)

In [ ]:
hourly_complete_summary = (
    hour_complete_days
    .groupby('hr')
    .agg(
        observed_records_complete=('datetime', 'count'),
        total_average_complete=('cnt', 'mean'),
        casual_average_complete=('casual', 'mean'),
        registered_average_complete=('registered', 'mean')
    )
    .reindex(range(24))
    .reset_index()
)

In [ ]:
hourly_sensitivity_summary = (
    hourly_all_summary
    .merge(
        hourly_complete_summary,
        on='hr',
        how='left'
    )
)

In [ ]:
hourly_sensitivity_summary[
    'total_difference_pct'
] = (
    (
        hourly_sensitivity_summary[
            'total_average_complete'
        ]
        - hourly_sensitivity_summary[
            'total_average_all'
        ]
    )
    / hourly_sensitivity_summary[
        'total_average_all'
    ]
    * 100
)

hourly_sensitivity_summary[
    'casual_difference_pct'
] = (
    (
        hourly_sensitivity_summary[
            'casual_average_complete'
        ]
        - hourly_sensitivity_summary[
            'casual_average_all'
        ]
    )
    / hourly_sensitivity_summary[
        'casual_average_all'
    ]
    * 100
)

hourly_sensitivity_summary[
    'registered_difference_pct'
] = (
    (
        hourly_sensitivity_summary[
            'registered_average_complete'
        ]
        - hourly_sensitivity_summary[
            'registered_average_all'
        ]
    )
    / hourly_sensitivity_summary[
        'registered_average_all'
    ]
    * 100
)

In [ ]:
hourly_sensitivity_display = (
    hourly_sensitivity_summary
    .round({
        'total_average_all': 2,
        'total_average_complete': 2,
        'casual_average_all': 2,
        'casual_average_complete': 2,
        'registered_average_all': 2,
        'registered_average_complete': 2,
        'total_difference_pct': 2,
        'casual_difference_pct': 2,
        'registered_difference_pct': 2
    })
)

display(hourly_sensitivity_display)

In [ ]:
peak_sensitivity_rows = []

sensitivity_datasets = {
    'all hourly records': hourly_all_summary,
    'complete days only': hourly_complete_summary
}

sensitivity_metrics = {
    'total demand': (
        'total_average_all',
        'total_average_complete'
    ),
    'casual users': (
        'casual_average_all',
        'casual_average_complete'
    ),
    'registered users': (
        'registered_average_all',
        'registered_average_complete'
    )
}

for sample_name, summary_data in sensitivity_datasets.items():
    for demand_measure, columns in sensitivity_metrics.items():

        if sample_name == 'all hourly records':
            value_column = columns[0]
        else:
            value_column = columns[1]

        peak_row = summary_data.loc[
            summary_data[value_column].idxmax()
        ]

        peak_sensitivity_rows.append({
            'analysis_sample': sample_name,
            'demand_measure': demand_measure,
            'peak_hour': int(peak_row['hr']),
            'peak_average_rentals': peak_row[value_column]
        })

peak_hour_sensitivity = pd.DataFrame(
    peak_sensitivity_rows
)

peak_hour_sensitivity[
    'peak_average_rentals'
] = (
    peak_hour_sensitivity[
        'peak_average_rentals'
    ]
    .round(2)
)

display(peak_hour_sensitivity)

In [ ]:
hourly_sensitivity_plot = (
    hourly_sensitivity_summary
    .set_index('hr')
    [
        [
            'total_average_all',
            'total_average_complete'
        ]
    ]
)

ax = hourly_sensitivity_plot.plot(
    kind='line',
    marker='o',
    figsize=(11, 5)
)

ax.set_title(
    'Hourly Demand Sensitivity to Incomplete Days'
)

ax.set_xlabel('Hour of Day')
ax.set_ylabel('Average Hourly Rentals')

ax.set_xticks(
    range(24)
)

ax.legend(
    [
        'All Hourly Records',
        'Complete Days Only'
    ],
    title='Analysis Sample'
)

plt.tight_layout()

plt.savefig(
    figures_output_dir
    / 'hourly_demand_sensitivity_incomplete_days.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
sensitivity_sample_summary.to_csv(
    tables_output_dir
    / 'sensitivity_sample_summary.csv',
    index=False
)

humidity_sensitivity_summary.to_csv(
    tables_output_dir
    / 'humidity_sensitivity_summary.csv',
    index=False
)

hourly_sensitivity_summary.to_csv(
    tables_output_dir
    / 'hourly_demand_sensitivity_summary.csv',
    index=False
)

peak_hour_sensitivity.to_csv(
    tables_output_dir
    / 'peak_hour_sensitivity.csv',
    index=False
)

### Data quality sensitivity analysis result

Removing the zero-humidity observation changed the examined correlation coefficients by no more than 0.0140.

Restricting the hourly analysis to complete dates changed average hourly demand estimates by up to 6.97%, but the identified peak hours for total, casual, and registered demand remained unchanged.

The main conclusions about peak-demand timing are therefore stable, while exact hourly demand magnitudes should still be interpreted with awareness of incomplete-date coverage.

## 10. EDA Findings and Operational Implications

This section consolidates the main exploratory findings and translates them into cautious operational implications.

All evidence reported below is derived from the calculated tables in the preceding sections. Descriptive associations are not interpreted as causal effects.

In [ ]:
peak_month_by_year = (
    monthly_demand_summary.loc[
        monthly_demand_summary
        .groupby('year')[
            'average_daily_rentals'
        ]
        .idxmax(),
        [
            'year',
            'month_name',
            'average_daily_rentals'
        ]
    ]
    .reset_index(drop=True)
)

peak_month_by_year[
    'average_daily_rentals'
] = (
    peak_month_by_year[
        'average_daily_rentals'
    ]
    .round(2)
)

display(peak_month_by_year)

In [ ]:
peak_season_by_year = (
    seasonal_demand_summary.loc[
        seasonal_demand_summary
        .groupby('year')[
            'average_daily_rentals'
        ]
        .idxmax(),
        [
            'year',
            'season_label',
            'average_daily_rentals'
        ]
    ]
    .reset_index(drop=True)
)

peak_season_by_year[
    'average_daily_rentals'
] = (
    peak_season_by_year[
        'average_daily_rentals'
    ]
    .round(2)
)

display(peak_season_by_year)

In [ ]:
peak_hour_stability = (
    peak_hour_sensitivity
    .pivot(
        index='demand_measure',
        columns='analysis_sample',
        values='peak_hour'
    )
    .reset_index()
)

peak_hour_stability['peak_hour_stable'] = (
    peak_hour_stability[
        'all hourly records'
    ]
    ==
    peak_hour_stability[
        'complete days only'
    ]
)

display(peak_hour_stability)

In [ ]:
dominant_segment_row = user_segment_summary.loc[
    user_segment_summary[
        'total_rentals'
    ].idxmax()
]

dominant_segment = (
    dominant_segment_row[
        'user_segment'
    ]
)

dominant_segment_share = (
    dominant_segment_row[
        'share_pct'
    ]
)

average_daily_growth = (
    annual_growth_summary.loc[
        annual_growth_summary[
            'metric'
        ] == 'average_daily_rentals',
        'growth_pct'
    ]
    .iloc[0]
)

In [ ]:
peak_hours = (
    peak_hour_summary
    .set_index('demand_measure')[
        'peak_hour'
    ]
)

total_peak_hour = int(
    peak_hours['total demand']
)

casual_peak_hour = int(
    peak_hours['casual users']
)

registered_peak_hour = int(
    peak_hours['registered users']
)

In [ ]:
casual_share_by_day_type = (
    segment_mix_by_day_type
    .set_index('day_type')[
        'casual_share_pct'
    ]
)

working_day_casual_share = (
    casual_share_by_day_type[
        'working_day'
    ]
)

non_working_day_casual_share = (
    casual_share_by_day_type[
        'non_working_day'
    ]
)

In [ ]:
weather_summary_for_selection = (
    weather_correlation_summary.loc[
        weather_correlation_summary[
            'weather_variable'
        ].isin(
            [
                'temp_c',
                'humidity_pct',
                'windspeed_original_scale'
            ]
        )
    ]
    .copy()
)

weather_summary_for_selection[
    'absolute_spearman'
] = (
    weather_summary_for_selection[
        'spearman_correlation'
    ]
    .abs()
)

strongest_weather_row = (
    weather_summary_for_selection.loc[
        weather_summary_for_selection[
            'absolute_spearman'
        ].idxmax()
    ]
)

In [ ]:
maximum_humidity_correlation_change = (
    humidity_sensitivity_summary[
        [
            'pearson_change',
            'spearman_change'
        ]
    ]
    .abs()
    .max()
    .max()
)

maximum_hourly_total_difference = (
    hourly_sensitivity_summary[
        'total_difference_pct'
    ]
    .abs()
    .max()
)

all_peak_hours_stable = (
    peak_hour_stability[
        'peak_hour_stable'
    ]
    .all()
)

In [ ]:
eda_key_metrics = pd.DataFrame({
    'metric': [
        'dominant user segment',
        'dominant segment share',
        'average daily demand growth',
        'total demand peak hour',
        'casual user peak hour',
        'registered user peak hour',
        'casual share on working days',
        'casual share on non-working days',
        'strongest selected weather association',
        'maximum humidity correlation change',
        'maximum hourly sensitivity difference',
        'all peak hours stable'
    ],

    'value': [
        dominant_segment,

        f'{dominant_segment_share:.2f}%',

        f'{average_daily_growth:.2f}%',

        f'{total_peak_hour}:00',

        f'{casual_peak_hour}:00',

        f'{registered_peak_hour}:00',

        f'{working_day_casual_share:.2f}%',

        f'{non_working_day_casual_share:.2f}%',

        (
            f"{strongest_weather_row['weather_variable']} "
            f"with {strongest_weather_row['demand_variable']} "
            f"(Spearman = "
            f"{strongest_weather_row['spearman_correlation']:.3f})"
        ),

        f'{maximum_humidity_correlation_change:.4f}',

        f'{maximum_hourly_total_difference:.2f}%',

        str(all_peak_hours_stable)
    ]
})

display(eda_key_metrics)

In [ ]:
if all_peak_hours_stable:
    quality_evidence = (
        'Peak-demand hours remained unchanged '
        'after incomplete dates were excluded.'
    )

    quality_implication = (
        'The main hourly peak findings appear '
        'operationally stable.'
    )

else:
    quality_evidence = (
        'At least one peak-demand hour changed '
        'after incomplete dates were excluded.'
    )

    quality_implication = (
        'Hourly peak conclusions should be used '
        'with additional caution.'
    )   

In [ ]:
peak_month_text = '; '.join(
    [
        (
            f"{int(row['year'])}: "
            f"{row['month_name']} "
            f"({row['average_daily_rentals']:.2f})"
        )
        for _, row in peak_month_by_year.iterrows()
    ]
)

peak_season_text = '; '.join(
    [
        (
            f"{int(row['year'])}: "
            f"{row['season_label']} "
            f"({row['average_daily_rentals']:.2f})"
        )
        for _, row in peak_season_by_year.iterrows()
    ]
)

In [ ]:
operational_implications = pd.DataFrame({
    'evidence_area': [
        'user composition',
        'annual demand change',
        'monthly and seasonal demand',
        'hourly demand',
        'working and non-working days',
        'weather conditions',
        'data quality sensitivity'
    ],

    'data_evidence': [
        (
            f'{dominant_segment} users represented '
            f'{dominant_segment_share:.2f}% '
            f'of recorded rentals.'
        ),

        (
            f'Average daily rentals changed by '
            f'{average_daily_growth:.2f}% '
            f'from 2011 to 2012.'
        ),

        (
            f'Peak months — {peak_month_text}. '
            f'Peak seasons — {peak_season_text}.'
        ),

        (
            f'Peak hours were {total_peak_hour}:00 '
            f'for total demand, '
            f'{casual_peak_hour}:00 for casual users, '
            f'and {registered_peak_hour}:00 '
            f'for registered users.'
        ),

        (
            f'Casual users represented '
            f'{working_day_casual_share:.2f}% '
            f'of working-day rentals and '
            f'{non_working_day_casual_share:.2f}% '
            f'of non-working-day rentals.'
        ),

        (
            f'The strongest selected Spearman '
            f'association was between '
            f"{strongest_weather_row['weather_variable']} "
            f"and {strongest_weather_row['demand_variable']} "
            f"({strongest_weather_row['spearman_correlation']:.3f})."
        ),

        quality_evidence
    ],

    'operational_implication': [
        (
            'Demand planning can distinguish between '
            'casual and registered user needs rather '
            'than treating all rentals as one market.'
        ),

        (
            'Capacity and service planning should account '
            'for the change in demand level between years.'
        ),

        (
            'Higher-demand periods can inform capacity '
            'planning, while lower-demand periods may be '
            'more suitable for planned maintenance.'
        ),

        (
            'Bike availability, station readiness, and '
            'rebalancing can be scheduled before the '
            'identified peak periods.'
        ),

        (
            'Working and non-working days should use '
            'different operating profiles because user '
            'composition is not identical.'
        ),

        (
            'Weather information can support short-term '
            'planning, but it should be combined with '
            'calendar and seasonal information.'
        ),

        quality_implication
    ],

    'interpretation_limit': [
        (
            'Rental records do not identify unique users '
            'or individual customer motives.'
        ),

        (
            'The dataset does not explain what caused '
            'the annual change.'
        ),

        (
            'Month and season overlap with weather, '
            'holidays, and annual growth.'
        ),

        (
            'Trip purpose is not recorded, so commuting '
            'and leisure interpretations remain inferred.'
        ),

        (
            'Non-working days combine weekends and '
            'official holidays.'
        ),

        (
            'Association does not establish causation, '
            'and weather variables may overlap.'
        ),

        (
            'Sensitivity analysis evaluates only the '
            'documented data-quality issues.'
        )
    ]
})

display(operational_implications)

In [ ]:
peak_month_by_year.to_csv(
    tables_output_dir
    / 'peak_month_by_year.csv',
    index=False
)

peak_season_by_year.to_csv(
    tables_output_dir
    / 'peak_season_by_year.csv',
    index=False
)

peak_hour_stability.to_csv(
    tables_output_dir
    / 'peak_hour_stability.csv',
    index=False
)

eda_key_metrics.to_csv(
    tables_output_dir
    / 'eda_key_metrics.csv',
    index=False
)

operational_implications.to_csv(
    tables_output_dir
    / 'operational_implications.csv',
    index=False
)

### Exploratory analysis completion

The exploratory analysis identified temporal, calendar, user-segment, and weather-related patterns in bike-sharing demand.

The final findings were generated directly from the calculated analysis outputs. Operational implications were separated from descriptive evidence, and each implication was accompanied by an interpretation limitation.

The exploratory findings provide a structured basis for statistical validation and modeling in JMP.